#Apache Spark Basics-Retail Sales Analysis

---

##Step 2-Spark Session


In [0]:
# In Databricks,SparkSession is pre-built just verify it works
print("Spark is ready")
print(f"Spark version:{spark.version}")

Spark is ready
Spark version:4.1.0


## Step 3-Load Data

In [0]:
# created via "Create or modify table" 
# No file path needed spark reads it from the catalog
df = spark.read.table("sales_data")
print(f"Data loaded")
print(f"Total rows:{df.count()}")
print(f"Total columns:{len(df.columns)}")

Data loaded
Total rows:510
Total columns:10


In [0]:
#view first 5 rows
print("First 5 rows")
df.show(5)

First 5 rows
+-----------+-------------+---+------+-----------+------+-----------+--------+----------+--------+
|customer_id|customer_name|age|gender|       city|region|   category|quantity|unit_price| revenue|
+-----------+-------------+---+------+-----------+------+-----------+--------+----------+--------+
|        164| Customer_164| 24|  Male|  Ahmedabad|  West|    Grocery|       8|    2206.6| 17652.8|
|        173| Customer_173| 29|Female|      Patna|  East|Electronics|       2|   1276.33| 2552.66|
|        466| Customer_466| 35|  Male|Bhubaneswar|  East|     Sports|       3|   1894.48| 5683.44|
|        441| Customer_441| 55| Other|       Pune|  West|     Sports|      19|   4089.27|77696.13|
|        254| Customer_254| 46| Other|    Chennai| South|    Grocery|       9|   4772.47|42952.23|
+-----------+-------------+---+------+-----------+------+-----------+--------+----------+--------+
only showing top 5 rows


In [0]:
#view column names
print("Column names:",df.columns)

Column names: ['customer_id', 'customer_name', 'age', 'gender', 'city', 'region', 'category', 'quantity', 'unit_price', 'revenue']


In [0]:
#view schema
print("Schema")
df.printSchema()

Schema
root
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- revenue: double (nullable = true)



##Step 4-Data Cleaning

In [0]:
#check null (missing) values in every column
from pyspark.sql import functions as F
print("Null counts per column")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

Null counts per column
+-----------+-------------+---+------+----+------+--------+--------+----------+-------+
|customer_id|customer_name|age|gender|city|region|category|quantity|unit_price|revenue|
+-----------+-------------+---+------+----+------+--------+--------+----------+-------+
|          0|            0| 11|     0|   0|     0|       0|       0|         0|      9|
+-----------+-------------+---+------+----+------+--------+--------+----------+-------+



In [0]:
#remove duplicate rows
before=df.count()
df=df.dropDuplicates()
after=df.count()
print(f"Rows before:{before}")
print(f"Rows after:{after}")
print(f"Duplicates removed:{before-after}")

Rows before:510
Rows after:500
Duplicates removed:10


In [0]:
#fill missing age values with mean age
mean_age=int(df.select(F.mean("age")).collect()[0][0])
df=df.fillna({"age": mean_age})
print(f"Filled missing age values with mean:{mean_age}")
#drop rows where revenue is null
df=df.dropna(subset=["revenue"])
print(f"Rows after dropping null revenue:{df.count()}")

Filled missing age values with mean:41
Rows after dropping null revenue:491


In [0]:
#verify no nulls remain
print("Null counts after cleaning")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

Null counts after cleaning
+-----------+-------------+---+------+----+------+--------+--------+----------+-------+
|customer_id|customer_name|age|gender|city|region|category|quantity|unit_price|revenue|
+-----------+-------------+---+------+----+------+--------+--------+----------+-------+
|          0|            0|  0|     0|   0|     0|       0|       0|         0|      0|
+-----------+-------------+---+------+----+------+--------+--------+----------+-------+



##Step 5-Filter Data

In [0]:
#filter customers aged 25 to 40
df_young=df.filter((F.col("age")>= 25)&(F.col("age")<= 40))
print(f"Customers aged 25–40:{df_young.count()}")
df_young.show(5)

Customers aged 25–40:148
+-----------+-------------+---+------+-----------+------+-----------+--------+----------+--------+
|customer_id|customer_name|age|gender|       city|region|   category|quantity|unit_price| revenue|
+-----------+-------------+---+------+-----------+------+-----------+--------+----------+--------+
|        173| Customer_173| 29|Female|      Patna|  East|Electronics|       2|   1276.33| 2552.66|
|        466| Customer_466| 35|  Male|Bhubaneswar|  East|     Sports|       3|   1894.48| 5683.44|
|        234| Customer_234| 25| Other|     Mumbai|  West|    Grocery|      14|   2939.33|41150.62|
|        136| Customer_136| 25| Other|    Kolkata|  East|     Sports|      15|    503.78|  7556.7|
|        293| Customer_293| 39| Other| Chandigarh| North|     Sports|      19|   4887.78|92867.82|
+-----------+-------------+---+------+-----------+------+-----------+--------+----------+--------+
only showing top 5 rows


In [0]:
#filter electronics category only
df_electronics=df.filter(F.col("category")=="Electronics")
print(f"Electronics orders:{df_electronics.count()}")
df_electronics.show(5)

Electronics orders:82
+-----------+-------------+---+------+---------+-------+-----------+--------+----------+--------+
|customer_id|customer_name|age|gender|     city| region|   category|quantity|unit_price| revenue|
+-----------+-------------+---+------+---------+-------+-----------+--------+----------+--------+
|        173| Customer_173| 29|Female|    Patna|   East|Electronics|       2|   1276.33| 2552.66|
|        314| Customer_314| 41|  Male|Ahmedabad|   West|Electronics|      18|   2989.41|53809.38|
|        268| Customer_268| 60|  Male|Hyderabad|  South|Electronics|      14|   3093.87|43314.18|
|        361| Customer_361| 45|Female|Ahmedabad|   West|Electronics|       8|    336.63| 2693.04|
|        367| Customer_367| 18|  Male|   Bhopal|Central|Electronics|       8|    277.61| 2220.88|
+-----------+-------------+---+------+---------+-------+-----------+--------+----------+--------+
only showing top 5 rows


In [0]:
#filter south region only
df_south=df.filter(F.col("region")=="South")
print(f"South region orders:{df_south.count()}")
df_south.show(5)

South region orders:99
+-----------+-------------+---+------+---------+------+-----------+--------+----------+--------+
|customer_id|customer_name|age|gender|     city|region|   category|quantity|unit_price| revenue|
+-----------+-------------+---+------+---------+------+-----------+--------+----------+--------+
|        254| Customer_254| 46| Other|  Chennai| South|    Grocery|       9|   4772.47|42952.23|
|        268| Customer_268| 60|  Male|Hyderabad| South|Electronics|      14|   3093.87|43314.18|
|        120| Customer_120| 47|Female|  Chennai| South|     Sports|       9|   3350.86|30157.74|
|        287| Customer_287| 41| Other|  Chennai| South|   Clothing|      17|   2307.25|39223.25|
|        452| Customer_452| 24|  Male|  Chennai| South|    Grocery|      13|   2541.52|33039.76|
+-----------+-------------+---+------+---------+------+-----------+--------+----------+--------+
only showing top 5 rows


##Step 6-Transform Data

In [0]:
#rename column unit_price to price_per_unit
df=df.withColumnRenamed("unit_price", "price_per_unit")
print("Renamed unit_price to price_per_unit")
print("Columns now:",df.columns)

Renamed unit_price to price_per_unit
Columns now: ['customer_id', 'customer_name', 'age', 'gender', 'city', 'region', 'category', 'quantity', 'price_per_unit', 'revenue']


In [0]:
#add new column discounted_revenue(10% discount applied)
df=df.withColumn("discounted_revenue",F.round(F.col("revenue")*0.90,2))
df.select("customer_name","revenue","discounted_revenue").show(5)

+-------------+--------+------------------+
|customer_name| revenue|discounted_revenue|
+-------------+--------+------------------+
| Customer_164| 17652.8|          15887.52|
| Customer_173| 2552.66|           2297.39|
| Customer_466| 5683.44|            5115.1|
| Customer_441|77696.13|          69926.52|
| Customer_254|42952.23|          38657.01|
+-------------+--------+------------------+
only showing top 5 rows


In [0]:
#add age_group column based on age ranges
df = df.withColumn("age_group",F.when(F.col("age") < 25, "Youth")
.when((F.col("age") >= 25) & (F.col("age") < 40), "Adult")
.otherwise("Senior"))
df.select("customer_name", "age", "age_group").show(5)

+-------------+---+---------+
|customer_name|age|age_group|
+-------------+---+---------+
| Customer_164| 24|    Youth|
| Customer_173| 29|    Adult|
| Customer_466| 35|    Adult|
| Customer_441| 55|   Senior|
| Customer_254| 46|   Senior|
+-------------+---+---------+
only showing top 5 rows


##Step 7-Basic Aggregation

In [0]:
#total rows,average age,revenue stats
print(f"Total records:{df.count()}")
df.select(F.round(F.avg("age"), 1).alias("avg_age"),
F.round(F.avg("revenue"), 2).alias("avg_revenue"),
F.round(F.min("revenue"), 2).alias("min_revenue"),
F.round(F.max("revenue"), 2).alias("max_revenue")).show()

Total records:491
+-------+-----------+-----------+-----------+
|avg_age|avg_revenue|min_revenue|max_revenue|
+-------+-----------+-----------+-----------+
|   41.8|   25298.93|     191.74|    96354.0|
+-------+-----------+-----------+-----------+



In [0]:
#summary statistics for numeric columns
df.select("age","quantity","price_per_unit","revenue").describe().show()

+-------+------------------+------------------+------------------+------------------+
|summary|               age|          quantity|    price_per_unit|           revenue|
+-------+------------------+------------------+------------------+------------------+
|  count|               491|               491|               491|               491|
|   mean| 41.78411405295316|10.201629327902241|2507.3364358452127|25298.926395112037|
| stddev|13.391827852989605| 5.864288097535222|1404.0159444804144| 21431.30307547816|
|    min|                18|                 1|             95.87|            191.74|
|    max|                65|                20|            4993.0|           96354.0|
+-------+------------------+------------------+------------------+------------------+



##Step 8-GroupBy Analysis

In [0]:
#revenue by Region
print("Revenue by Region")
df.groupBy("region").agg(F.count("customer_id").alias("orders"),
F.round(F.sum("revenue"),2).alias("total_revenue"),
F.round(F.avg("revenue"),2).alias("avg_revenue")
).orderBy(F.desc("total_revenue")).show()

Revenue by Region
+-------+------+-------------+-----------+
| region|orders|total_revenue|avg_revenue|
+-------+------+-------------+-----------+
|   West|   110|   2674544.23|   24314.04|
|  South|    99|   2630503.09|   26570.74|
|  North|    95|   2547334.15|   26814.04|
|Central|   100|   2415074.51|   24150.75|
|   East|    87|   2154316.88|   24762.26|
+-------+------+-------------+-----------+



In [0]:
#revenue by Category
print("Revenue by Category")
df.groupBy("category").agg(F.count("customer_id").alias("orders"),
F.round(F.sum("revenue"),2).alias("total_revenue"),
F.round(F.avg("quantity"),2).alias("avg_qty")
).orderBy(F.desc("total_revenue")).show()

Revenue by Category
+-----------+------+-------------+-------+
|   category|orders|total_revenue|avg_qty|
+-----------+------+-------------+-------+
|     Sports|   106|   2744896.02|   9.93|
|   Clothing|   105|   2724456.29|  10.51|
|      Books|    97|   2460954.27|  10.68|
|    Grocery|   101|    2443111.5|   9.66|
|Electronics|    82|   2048354.78|  10.24|
+-----------+------+-------------+-------+



In [0]:
#customer count by age group
print("Orders by age group")
df.groupBy("age_group").agg(F.count("customer_id").alias("customers"),
F.round(F.avg("revenue"),2).alias("avg_revenue")
).orderBy("age_group").show()

Orders by age group
+---------+---------+-----------+
|age_group|customers|avg_revenue|
+---------+---------+-----------+
|    Adult|      138|   25682.78|
|   Senior|      284|   25591.82|
|    Youth|       69|   23325.69|
+---------+---------+-----------+



In [0]:
#top 5 cities by revenue
print("Top 5 cities")
df.groupBy("city").agg(F.round(F.sum("revenue"),2).alias("total_revenue")
).orderBy(F.desc("total_revenue")).show(5)

Top 5 cities
+----------+-------------+
|      city|total_revenue|
+----------+-------------+
|    Mumbai|   1091463.21|
|    Raipur|   1081479.28|
| Hyderabad|   1007111.37|
|Chandigarh|    956761.96|
|     Delhi|    922091.22|
+----------+-------------+
only showing top 5 rows


#Cleaned Dataset

In [0]:
df.count()

491

##Step 10-Full Pipeline

In [0]:
from pyspark.sql import functions as F

# Load the data
df = spark.read.table("sales_data")

# Remove duplicate rows
df = df.dropDuplicates()

# Fill missing age values with the average age
avg_age = int(df.select(F.mean("age")).first()[0])
df = df.fillna({"age": avg_age})

# Remove rows where revenue is missing
df = df.dropna(subset=["revenue"])

# Filter required records
df = df.filter(
    (F.col("category").isin("Electronics", "Sports")) &
    (F.col("age").between(20, 55))
)

# Rename column
df = df.withColumnRenamed("unit_price", "price_per_unit")

# Add profit column
df = df.withColumn(
    "profit_margin",
    F.round(F.col("revenue") * 0.20, 2)
)

# Create age groups
df = df.withColumn(
    "age_group",
    F.when(F.col("age") < 25, "Youth")
     .when(F.col("age") < 40, "Adult")
     .otherwise("Senior")
)

# Aggregate the data
final_df = df.groupBy("region", "category").agg(
    F.count("customer_id").alias("total_orders"),
    F.sum("revenue").alias("total_revenue"),
    F.avg("revenue").alias("average_revenue"),
    F.sum("profit_margin").alias("total_profit"),
    F.avg("quantity").alias("average_quantity")
)

# Sort by revenue
final_df = final_df.orderBy(F.desc("total_revenue"))

# Display the result
display(final_df)

# Save the output
final_df.write.mode("overwrite").saveAsTable("results_output")

print("Pipeline completed successfully.")

region,category,total_orders,total_revenue,average_revenue,total_profit,average_quantity
North,Sports,18,564081.16,31337.842222222225,112816.21,10.0
East,Electronics,16,481563.37,30097.710625,96312.68,9.5
West,Sports,16,426053.61999999994,26628.351249999996,85210.72,10.5
East,Sports,21,405814.24,19324.487619047617,81162.84,8.952380952380953
Central,Sports,15,397742.11,26516.140666666666,79548.41999999998,10.066666666666666
South,Sports,16,394499.51000000007,24656.219375000004,78899.9,9.4375
West,Electronics,14,361887.28,25849.09142857143,72377.46999999999,9.714285714285714
South,Electronics,12,341615.67000000004,28467.972500000003,68323.14000000001,13.0
North,Electronics,12,236284.83999999997,19690.403333333332,47256.96,9.333333333333334
Central,Electronics,10,228559.24,22855.924,45711.850000000006,6.9


Pipeline completed successfully.
